# UC-02: Invoice Three-Way Match — Model Explanation

SHAP analysis, feature importance, error analysis, and business interpretation
of the trained invoice match prediction model.

## 1. Load Model & Data

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[2]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")

from ml.common.db_config import load_tables
from ml.data_processing.python.uc02_preprocessing import UC02_TABLES
from ml.uc_02_invoice_match.feature_engineering.feature_functions import (
    build_uc02_features, prepare_feature_matrix,
)

In [ ]:
# Load model
artifact = joblib.load("best_model.joblib")
model = artifact["model"]
feature_columns = artifact["feature_columns"]
print(f"Model: {artifact['model_name']}, CV F1: {artifact['cv_f1']:.4f}")

# Load data
csv_dir = project_root / "output" / "csv"
tables = load_tables("csv", UC02_TABLES, csv_dir=csv_dir)
feature_df = build_uc02_features(tables, leave_one_out=True)
X, y = prepare_feature_matrix(feature_df, target="binary")

# Align columns to model
X = X[feature_columns]
print(f"Features: {X.shape}, Target: {dict(y.value_counts())}")

## 2. Native Feature Importance

In [ ]:
# Get feature importance (works for tree models and LR)
if hasattr(model, "feature_importances_"):
    importances = model.feature_importances_
    importance_type = "gain"
elif hasattr(model, "named_steps"):
    if hasattr(model.named_steps.get("lr", None), "coef_"):
        importances = np.abs(model.named_steps["lr"].coef_[0])
        importance_type = "coefficient magnitude"
    else:
        importances = np.zeros(len(feature_columns))
        importance_type = "unknown"
else:
    importances = np.zeros(len(feature_columns))
    importance_type = "unknown"

importance_df = pd.DataFrame({
    "feature": feature_columns,
    "importance": importances,
}).sort_values("importance", ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=importance_df, x="importance", y="feature", ax=ax, color="steelblue")
ax.set_title(f"Top 20 Features by {importance_type}")
plt.tight_layout()
plt.show()

## 3. SHAP Analysis

SHAP (SHapley Additive exPlanations) provides consistent, locally accurate feature
attributions. For tree models, TreeExplainer provides exact SHAP values.

In [ ]:
# Create SHAP explainer
if hasattr(model, "feature_importances_"):
    explainer = shap.TreeExplainer(model)
else:
    # For Pipeline (LR), use KernelExplainer on a sample
    background = shap.sample(X, 50)
    explainer = shap.KernelExplainer(model.predict_proba, background)

shap_values = explainer.shap_values(X)

# For binary classification, shap_values might be a list [class_0, class_1]
if isinstance(shap_values, list):
    shap_vals = shap_values[1]  # class 1 = ANY_VARIANCE
else:
    shap_vals = shap_values

print(f"SHAP values shape: {shap_vals.shape}")

In [ ]:
# Summary beeswarm plot
shap.summary_plot(shap_vals, X, max_display=20, show=False)
plt.title("SHAP Summary — Feature Impact on Variance Prediction")
plt.tight_layout()
plt.show()

In [ ]:
# SHAP bar plot (mean absolute SHAP)
shap.summary_plot(shap_vals, X, plot_type="bar", max_display=20, show=False)
plt.title("Mean |SHAP| — Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
# Dependence plots for top 5 features
mean_abs_shap = np.abs(shap_vals).mean(axis=0)
top5_idx = np.argsort(mean_abs_shap)[::-1][:5]
top5_features = [feature_columns[i] for i in top5_idx]

fig, axes = plt.subplots(1, 5, figsize=(25, 4))
for i, (feat, ax) in enumerate(zip(top5_features, axes)):
    shap.dependence_plot(feat, shap_vals, X, ax=ax, show=False)
    ax.set_title(feat)
plt.tight_layout()
plt.show()

## 4. Per-Prediction Explanations

In [ ]:
# Select 3 representative examples
preds = model.predict(X)
probas = model.predict_proba(X)

# 1. Correct full match (high confidence)
correct_match_mask = (y == 0) & (preds == 0)
correct_match_idx = correct_match_mask[correct_match_mask].index
if len(correct_match_idx) > 0:
    example_match = correct_match_idx[np.argmax(probas[correct_match_mask, 0])]

# 2. Correct variance (high confidence)
correct_var_mask = (y == 1) & (preds == 1)
correct_var_idx = correct_var_mask[correct_var_mask].index
if len(correct_var_idx) > 0:
    example_var = correct_var_idx[np.argmax(probas[correct_var_mask, 1])]

# 3. Misclassified
misclass_mask = y != preds
misclass_idx = misclass_mask[misclass_mask].index
example_misclass = misclass_idx[0] if len(misclass_idx) > 0 else 0

examples = {
    "Correct Match": example_match if len(correct_match_idx) > 0 else 0,
    "Correct Variance": example_var if len(correct_var_idx) > 0 else 0,
    "Misclassified": example_misclass,
}

for label, idx in examples.items():
    pos = list(X.index).index(idx)
    print(f"\n--- {label} (index={idx}) ---")
    print(f"True: {int(y.loc[idx])}, Predicted: {int(preds[pos])}, P(variance): {probas[pos, 1]:.3f}")
    shap.waterfall_plot(shap.Explanation(
        values=shap_vals[pos],
        base_values=explainer.expected_value if not isinstance(explainer.expected_value, list)
                     else explainer.expected_value[1],
        data=X.iloc[pos],
        feature_names=feature_columns,
    ), max_display=10, show=True)

## 5. Feature Interaction

In [ ]:
# SHAP interaction values (only for tree models)
if hasattr(model, "feature_importances_"):
    # Sample for speed
    X_sample = X.sample(min(100, len(X)), random_state=42)
    shap_interaction = explainer.shap_interaction_values(X_sample)

    if isinstance(shap_interaction, list):
        shap_interaction = shap_interaction[1]

    # Mean absolute interaction
    mean_interaction = np.abs(shap_interaction).mean(axis=0)
    np.fill_diagonal(mean_interaction, 0)  # Remove self-interactions

    # Top interactions
    interaction_df = pd.DataFrame(mean_interaction, index=feature_columns, columns=feature_columns)
    top_pairs = []
    for i in range(len(feature_columns)):
        for j in range(i+1, len(feature_columns)):
            top_pairs.append((feature_columns[i], feature_columns[j], mean_interaction[i, j]))
    top_pairs.sort(key=lambda x: x[2], reverse=True)

    print("Top 10 Feature Interactions:")
    for f1, f2, val in top_pairs[:10]:
        print(f"  {f1} x {f2}: {val:.4f}")
else:
    print("SHAP interaction values only available for tree models.")

## 6. Error Analysis

In [ ]:
# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y, preds)
ConfusionMatrixDisplay(cm, display_labels=["FULL_MATCH", "ANY_VARIANCE"]).plot(ax=axes[0])
axes[0].set_title("Confusion Matrix (Training Set)")

# Misclassified samples — distribution of key features
misclass_df = X.copy()
misclass_df["true"] = y.values
misclass_df["pred"] = preds
misclass_df["correct"] = misclass_df["true"] == misclass_df["pred"]

misclassified = misclass_df[~misclass_df["correct"]]
print(f"\nMisclassified: {len(misclassified)} / {len(misclass_df)} ({len(misclassified)/len(misclass_df)*100:.1f}%)")
print(f"  False Positives (predicted variance, actually match): {((misclass_df['pred']==1)&(misclass_df['true']==0)).sum()}")
print(f"  False Negatives (predicted match, actually variance): {((misclass_df['pred']==0)&(misclass_df['true']==1)).sum()}")

# Top feature differences between correct and misclassified
if len(misclassified) > 0 and len(top5_features) > 0:
    feat_to_show = top5_features[0]
    sns.boxplot(data=misclass_df, x="correct", y=feat_to_show, ax=axes[1])
    axes[1].set_title(f"{feat_to_show}: Correct vs Misclassified")
    axes[1].set_xticklabels(["Misclassified", "Correct"])

plt.tight_layout()
plt.show()

In [ ]:
# Classification report
print("Classification Report:")
print(classification_report(y, preds, target_names=["FULL_MATCH", "ANY_VARIANCE"]))

## 7. Business Interpretation

### Mapping SHAP to Business Insights

The model's predictions can be interpreted in business terms:

1. **Vendor quality/risk scores** — Vendors with lower quality scores and higher risk
   scores are more likely to produce invoice variances. This aligns with the data
   generation logic where quality_score directly influences match_rate.

2. **Historical vendor match rate** (LOO) — Past invoice matching performance is
   predictive of future outcomes. Vendors with consistently clean invoices continue
   to perform well.

3. **Contract coverage** — POs backed by contracts have agreed-upon prices,
   reducing the likelihood of price variances.

4. **GR quality indicators** — Goods receipt rejections and quality holds
   signal vendor quality issues that also manifest as invoice variances.

5. **PO characteristics** — Maverick and rush POs, which bypass normal procurement
   processes, are more prone to matching issues.

### Demo Scenario Narrative

"When a new invoice arrives from Vendor X, our model scores it for match risk.
For a high-risk invoice, the top contributing factors might be: the vendor's
historical 45% variance rate (vs. 23% average), a recent GR quality hold on
this PO, and the fact that this was a maverick purchase without a contract.
AP staff can prioritize reviewing this invoice before it enters the matching
queue, potentially catching issues hours or days earlier."

In [ ]:
# Natural language explanation for a sample prediction
sample_idx = 0
sample_proba = probas[sample_idx, 1]
sample_shap = shap_vals[sample_idx]
top_contrib = np.argsort(np.abs(sample_shap))[::-1][:5]

print(f"Invoice prediction: {'HIGH RISK' if sample_proba > 0.5 else 'LOW RISK'} ({sample_proba:.1%} variance probability)")
print(f"\nTop factors:")
for i, idx in enumerate(top_contrib, 1):
    feat = feature_columns[idx]
    val = X.iloc[sample_idx, idx]
    direction = 'increases' if sample_shap[idx] > 0 else 'decreases'
    print(f"  {i}. {feat} = {val:.2f} → {direction} risk by {abs(sample_shap[idx]):.3f}")